In [1]:
## Импортирую библиотеки для анализа данных
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [2]:
## Подключаюсь к базе movies: параметры (пароль) читаю из .env
load_dotenv()
url = (f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
       f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database()")).scalar())

movies


In [3]:
## Проверяю, что представление v_roi_movies читается из ноутбука
pd.read_sql("SELECT count(*) FROM v_roi_movies", engine)

,count
0,7004


In [4]:
## Загружаю представление v_roi_movies (гипотезы 2-5, окупаемость) в датафрейм
df_roi = pd.read_sql("SELECT * FROM v_roi_movies", engine)
## Проверяю размер: ожидаю (7004, 11)
df_roi.shape

(7004, 11)

In [5]:
## Загружаю представление v_rating_movies (гипотеза 1, рейтинг) в датафрейм
df_rating = pd.read_sql("SELECT * FROM v_rating_movies", engine)
## Проверяю размер: ожидаю (34076, 6)
df_rating.shape

(34076, 6)

In [6]:
## Проверяю типы столбцов df_roi до приведения
df_roi.dtypes

id                    int64
title                   str
release_date         object
release_month       float64
runtime               int64
budget                int64
revenue               int64
roi                 float64
certification_us        str
imdb_rating         float64
imdb_votes            int64
dtype: object

In [7]:
## Привожу столбцы к нужным типам: release_month - int, release_date - datetime
df_roi["release_month"] = df_roi.release_month.astype(int)
df_roi["release_date"] = pd.to_datetime(df_roi.release_date)
df_rating["release_date"] = pd.to_datetime(df_rating.release_date)
df_roi.dtypes 

id                          int64
title                         str
release_date        datetime64[s]
release_month               int64
runtime                     int64
budget                      int64
revenue                     int64
roi                       float64
certification_us              str
imdb_rating               float64
imdb_votes                  int64
dtype: object

In [ ]:
## Пропуски: df_roi.isna().sum() и то же для df_rating. 
## Ожидаю, что в df_roi пропуски только в certification_us (1929), а в df_rating их нет вообще.
print(df_roi.isna().sum())
print(df_rating.isna().sum())

id                     0
title                  0
release_date           0
release_month          0
runtime                0
budget                 0
revenue                0
roi                    0
certification_us    1929
imdb_rating            0
imdb_votes             0
dtype: int64
id              0
title           0
release_date    0
runtime         0
imdb_rating     0
imdb_votes      0
dtype: int64


In [9]:
## Сверяю медиану roi с SQL: ожидаю 1.5555019883040937
print(df_roi["roi"].median())

1.5555019883040937


In [10]:
## Смотрю на распределение длительности фильмов в df_rating
df_rating["runtime"].describe()

count    34076.000000
mean       104.920296
std         23.821843
min         60.000000
25%         90.000000
50%        100.000000
75%        115.000000
max        622.000000
Name: runtime, dtype: float64

In [11]:
## Смотрю 10 самых длинных фильмов: нет ли среди них многосерийных проектов.
df_rating.nlargest(10, "runtime")[["title", "release_date", "runtime", "imdb_rating", "imdb_votes"]]

,title,release_date,runtime,imdb_rating,imdb_votes
34070,Utopia,2013-01-15,622,8.4,56482
19591,Kingdom Hospital,2004-03-03,608,6.7,8407
33642,Paint Drying,2023-11-10,607,8.9,1672
14279,The Untold History Of The United States,2012-11-12,585,8.6,10162
11698,Tie Xi Qu: West of the Tracks,2004-06-09,551,8.2,1563
33362,The Chosen: Season 4,2024-02-01,546,8.8,1301
7701,Live 8,2005-11-07,540,7.7,1246
12304,America: The Story of Us,2010-09-14,540,7.3,2187
34073,Eastwick,2009-09-23,520,6.5,9252
19633,O.J.: Made in America,2016-05-20,467,8.9,24193


In [12]:
## Считаю количество фильмов с длительностью более 180, 240 и 300 минут
print(f"более 180 минут: { (df_rating['runtime'] > 180).sum() }")
print(f"более 240 минут: { (df_rating['runtime'] > 240).sum() }")
print(f"более 300 минут: { (df_rating['runtime'] > 300).sum() }")

более 180 минут: 210
более 240 минут: 48
более 300 минут: 26


In [13]:
## Смотрю фильмы длительностью 180-250 минут: где кончаются фильмы и начинаются сериалы
mask = df_rating["runtime"].between(180, 250)
df_rating[mask].sort_values("runtime", ascending=False)[["title", "release_date", "runtime", "imdb_rating", "imdb_votes"]].head(40)

,title,release_date,runtime,imdb_rating,imdb_votes
14234,"Norte, the End of History",2014-03-11,250,7.5,2486
14631,Vietnam in HD,2011-11-08,250,8.3,1831
5757,Philosophy of a Knife,2008-07-08,249,4.4,1849
12470,Extraordinary Stories,2008-10-04,242,7.8,1488
21296,Long Strange Trip,2017-05-25,242,8.2,2221
29013,Zack Snyder's Justice League,2021-03-18,242,7.9,480713
9616,WWE WrestleMania 22,2006-04-02,240,7.7,1590
9208,WWE Wrestlemania XXVI,2010-03-28,240,7.4,1312
7061,Never Sleep Again: The Elm Street Legacy,2010-05-04,240,8.5,5520
9823,WWE WrestleMania XXVII,2011-04-02,240,6.6,1282


In [14]:
## Помечаю фильмы длиннее 120 минут (ровно 120 относится к коротким)
df_rating["is_long"] = df_rating["runtime"] > 120

In [15]:
## Сравниваю рейтинг длинных и коротких фильмов: все фильмы
df_rating.groupby("is_long")["imdb_rating"].agg(["count", "mean", "median"])

,count,mean,median
is_long,,,
False,27715,5.958376,6.2
True,6361,6.629555,6.8


In [16]:
## Сравниваю рейтинг длинных и коротких фильмов: фильмы до 240 минут
df_rating[df_rating["runtime"] <= 240].groupby("is_long")["imdb_rating"].agg(["count", "mean", "median"])

,count,mean,median
is_long,,,
False,27715,5.958376,6.2
True,6313,6.621955,6.8


In [17]:
## Сравниваю рейтинг длинных и коротких фильмов: фильмы до 180 минут
df_rating[df_rating["runtime"] <= 180].groupby("is_long")["imdb_rating"].agg(["count", "mean", "median"])

,count,mean,median
is_long,,,
False,27715,5.958376,6.2
True,6151,6.605414,6.8


**Гипотеза 1: подтверждается.** 
Среди фильмов 2000-2024 с 1000+ голосами IMDb
длинные (> 120 мин, 6 361 фильм) оценены выше коротких (27 715): среднее 6,63
против 5,96, медиана 6,8 против 6,2. Проверка устойчивости: без фильмов длиннее
240 и 180 минут разница почти не меняется (0,66 и 0,65 пункта). Это связь, а не
причина: возможно длинные фильмы чаще крупные и престижные, а выборка ограничена
известными фильмами.

In [ ]:
## Оставляю только PG-13 и R для гипотезы 2
df_cert = df_roi[df_roi["certification_us"].isin(["PG-13", "R"])].copy()
df_cert.shape

(4009, 11)

In [19]:
## Сравниваю окупаемость PG-13 и R: количество, среднее и медиана roi
df_cert.groupby("certification_us")["roi"].agg(["count", "mean", "median"])

,count,mean,median
certification_us,,,
PG-13,1774,3.795890,2.045288
R,2235,9.548816,1.245057


In [20]:
## Смотрю, какие фильмы с рейтингом R имеют самый высокий roi
mask_cert = df_cert["certification_us"] == "R"
df_cert[mask_cert].nlargest(5, "roi")[["title", "certification_us", "budget", "revenue", "roi"]]

,title,certification_us,budget,revenue,roi
2327,Paranormal Activity,R,15000,193355800,12890.386667
6029,Animal World,R,125000,69885700,559.085600
23,Open Water,R,120000,54683487,455.695725
4738,The Gallows,R,100000,42964410,429.644100
6996,Demon Slayer: Kimetsu no Yaiba -To the Hashira...,R,200000,44354825,221.774125


In [21]:
## Проверка устойчивости: без фильмов с бюджетом от 1 млн и сборами меньше 100 тысяч
mask_susp = (df_cert["budget"] >= 1_000_000) & (df_cert["revenue"] < 100_000)
df_cert[~mask_susp].groupby("certification_us")["roi"].agg(["count", "median", "mean"])

,count,median,mean
certification_us,,,
PG-13,1753,2.078333,3.841250
R,2124,1.370598,10.047243


In [22]:
## Сравниваю медианы бюджета PG-13 и R
df_cert.groupby("certification_us")["budget"].median()

certification_us
PG-13    30000000.0
R        15000000.0
Name: budget, dtype: float64

In [ ]:
## Проверяю гипотезу 2 внутри бюджетных групп: ниже и выше медианы бюджета df_cert
budget_median = df_cert["budget"].median()
cmp_low = df_cert[df_cert["budget"] <= budget_median].groupby("certification_us")["roi"].agg(["count", "median"])
cmp_high = df_cert[df_cert["budget"] > budget_median].groupby("certification_us")["roi"].agg(["count", "median"])

print("Медиана бюджета:")
print(budget_median)
print("Фильмы с бюджетом ниже медианы: ")
print(cmp_low)
print("Фильмы с бюджетом выше медианы: ")
print(cmp_high)

Медиана бюджета:
20000000.0
Фильмы с бюджетом ниже медианы: 
                  count    median
certification_us                 
PG-13               695  2.150216
R                  1408  1.092684
Фильмы с бюджетом выше медианы: 
                  count    median
certification_us                 
PG-13              1079  2.013134
R                   827  1.504560


**Гипотеза 2: подтверждается по медиане.** Среди фильмов 2000-2024 с рейтингом PG-13
(1 774) и R (2 235) медиана окупаемости 2,05 против 1,25. Без 132 фильмов с бюджетом
от 1 млн и сборами меньше 100 тысяч: 2,08 против 1,37. Среднее у R выше (9,5 против
3,8) из-за нескольких хитов, поэтому вывод делается по медиане. Это связь, а не причина:
фильмы PG-13 и R могут отличаться бюджетом и жанром. Медиана бюджета у PG-13 30 млн, у R 15 млн. Внутри бюджетных групп (граница 20 млн) PG-13 тоже впереди: 2,15 против 1,09 при бюджете до 20 млн и 2,01 против 1,50 выше. Бюджет объясняет разницу лишь частично.